In [5]:
import geopandas as gpd
import pandas as pd
pd.set_option('display.max_columns', None)
import rasterio 
import glob
import numpy as np
from shapely.geometry import LineString, MultiLineString
import math
from pyproj import CRS

import plotly.express as px
import folium as fl
import string

import numpy as np, rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio import mask as rio_mask
import matplotlib.cm as mplcm
import matplotlib.colors as mplcolors
from branca.colormap import LinearColormap
from branca.element import Template, MacroElement
import branca
import plotly.graph_objects as go

from rapidfuzz import process, fuzz
import warnings
warnings.filterwarnings('ignore')

In [6]:
# File paths (edit to your rasters)
elev_tif   = r"D:\Work\ADB\adb-school-optimize\data\rasters\ALOS DEM.tif"
land_tif   = r"D:\Work\ADB\adb-school-optimize\data\rasters\Dynamic World LULC.tif"
flood_tif  = r"D:\Work\ADB\adb-school-optimize\data\rasters\PNG_flood_JRC.tif"
# flood_tif = "D:\Work\ADB\adb-school-optimize\data\PNG_flood_WRI.tif"
lumi_tif   = r"D:\Work\ADB\adb-school-optimize\data\luminosity\lumi_total_2025.tif"

landcover_palette = {
    0: "#419bdf",  # water
    1: "#397d49",  # trees
    2: "#88b053",  # grass
    3: "#7a87c6",  # flooded_vegetation
    4: "#e49635",  # crops 
    5: "#dfc35a",  # shrub
    6: "#c4281b",  # built-up
    7: "#a59b8f",  # bare
    8: "#b39fe1",  # snow_ice
}

# Define labels once (matching your palette codes)
landcover_labels = {
    0: "Water",
    1: "Trees",
    2: "Grass",
    3: "Flooded Vegetation",
    4: "Crops",
    5: "Shrub",
    6: "Built-up",
    7: "Bare",
    8: "Snow/Ice",
}

In [7]:
from branca.element import Template, MacroElement

def _crop_to_geom(src, geom_gdf):
    if geom_gdf is None:
        return src.read(1), src.transform, src.nodata, src.crs
    geom = geom_gdf.to_crs(src.crs).geometry.values
    arr, out_transform = rio_mask.mask(src, geom, crop=True, filled=True, nodata=src.nodata)
    return arr[0], out_transform, src.nodata, src.crs

def _to_wgs84(arr, transform, src_crs, dst_res=None, nodata=None, is_float=True):
    dst_crs = "EPSG:4326"
    h, w = arr.shape
    bounds = rio.transform.array_bounds(h, w, transform)
    new_transform, new_w, new_h = calculate_default_transform(
        src_crs, dst_crs, w, h, *bounds, resolution=dst_res
    )
    dst = np.full((new_h, new_w), nodata if nodata is not None else np.nan, dtype=arr.dtype)
    reproject(
        source=arr, destination=dst,
        src_transform=transform, src_crs=src_crs,
        dst_transform=new_transform, dst_crs=dst_crs,
        resampling=Resampling.bilinear if is_float else Resampling.nearest
    )
    return dst, new_transform

def _bounds_from_transform(arr, transform):
    h, w = arr.shape
    minx, miny, maxx, maxy = rio.transform.array_bounds(h, w, transform)
    return [[miny, minx], [maxy, maxx]]


def build_continuous_overlay(tif_path, geom_clip=None, cmap_name="viridis", dst_res=None):
    with rio.open(tif_path) as src:
        arr, tr, nd, src_crs = _crop_to_geom(src, geom_clip)
        arr_wgs, tr_wgs = _to_wgs84(
            arr, tr, src_crs, dst_res=dst_res, nodata=nd,
            is_float=np.issubdtype(arr.dtype, np.floating)
        )
    data = arr_wgs.astype("float32")
    mask = np.isfinite(data) & (data != nd)
    if not np.any(mask):
        vmin, vmax = float(np.nanmin(data)), float(np.nanmax(data))
    else:
        vmin, vmax = np.nanpercentile(data[mask], 2), np.nanpercentile(data[mask], 98)
    norm = mplcolors.Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = mplcm.get_cmap(cmap_name)
    rgba = (cmap(norm(data)) * 255).astype(np.uint8)
    alpha = rgba[..., 3].astype(np.float32) / 255.0
    alpha[~mask] = 0.0
    rgba[..., 3] = (alpha * 255).astype(np.uint8)
    bounds = _bounds_from_transform(data, tr_wgs)
    return rgba, bounds, vmin, vmax

def build_categorical_overlay(tif_path, class_to_color, geom_clip=None, dst_res=None, nodata=None):
    with rio.open(tif_path) as src:
        arr, tr, nd, src_crs = _crop_to_geom(src, geom_clip)
        if nodata is None: nodata = nd
        arr_wgs, tr_wgs = _to_wgs84(
            arr, tr, src_crs, dst_res=dst_res, nodata=nodata, is_float=False
        )
    h, w = arr_wgs.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    for cls, hex_color in class_to_color.items():
        mask = arr_wgs == cls
        if not mask.any(): continue
        r, g, b, _ = mplcolors.to_rgba(hex_color)
        rgba[mask, 0:3] = (np.array([r, g, b]) * 255).astype(np.uint8)
        rgba[mask, 3] = 160  # ~0.63 opacity
    rgba[(arr_wgs == nodata) | ~np.isfinite(arr_wgs)] = np.array([0, 0, 0, 0], dtype=np.uint8)
    bounds = _bounds_from_transform(arr_wgs, tr_wgs)
    return rgba, bounds


def build_continuous_overlay(tif_path, geom_clip=None, cmap_name="viridis", dst_res=None):
    with rio.open(tif_path) as src:
        arr, tr, nd, src_crs = _crop_to_geom(src, geom_clip)
        arr_wgs, tr_wgs = _to_wgs84(
            arr, tr, src_crs, dst_res=dst_res, nodata=nd,
            is_float=np.issubdtype(arr.dtype, np.floating)
        )
    data = arr_wgs.astype("float32")
    mask = np.isfinite(data) & (data != nd)
    if not np.any(mask):
        vmin, vmax = float(np.nanmin(data)), float(np.nanmax(data))
    else:
        vmin, vmax = np.nanpercentile(data[mask], 2), np.nanpercentile(data[mask], 98)
    norm = mplcolors.Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = mplcm.get_cmap(cmap_name)
    rgba = (cmap(norm(data)) * 255).astype(np.uint8)
    alpha = rgba[..., 3].astype(np.float32) / 255.0
    alpha[~mask] = 0.0
    rgba[..., 3] = (alpha * 255).astype(np.uint8)
    bounds = _bounds_from_transform(data, tr_wgs)
    return rgba, bounds, vmin, vmax

def build_categorical_overlay(tif_path, class_to_color, geom_clip=None, dst_res=None, nodata=None):
    with rio.open(tif_path) as src:
        arr, tr, nd, src_crs = _crop_to_geom(src, geom_clip)
        if nodata is None: nodata = nd
        arr_wgs, tr_wgs = _to_wgs84(
            arr, tr, src_crs, dst_res=dst_res, nodata=nodata, is_float=False
        )
    h, w = arr_wgs.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    for cls, hex_color in class_to_color.items():
        mask = arr_wgs == cls
        if not mask.any(): continue
        r, g, b, _ = mplcolors.to_rgba(hex_color)
        rgba[mask, 0:3] = (np.array([r, g, b]) * 255).astype(np.uint8)
        rgba[mask, 3] = 160  # ~0.63 opacity
    rgba[(arr_wgs == nodata) | ~np.isfinite(arr_wgs)] = np.array([0, 0, 0, 0], dtype=np.uint8)
    bounds = _bounds_from_transform(arr_wgs, tr_wgs)
    return rgba, bounds

import matplotlib.cm as mplcm
import matplotlib.colors as mplcolors
from branca.colormap import LinearColormap
from branca.element import Template, MacroElement

def _mpl_to_hex_list(cmap_name: str, n: int = 6):
    cmap = mplcm.get_cmap(cmap_name, n)
    return [mplcolors.to_hex(cmap(i)) for i in range(n)]

def add_colorbar(m, name: str, vmin: float, vmax: float, cmap_name: str = "viridis", position: str = "right"):
    colors = _mpl_to_hex_list(cmap_name, 8)
    colormap = LinearColormap(colors=colors, vmin=vmin, vmax=vmax)
    colormap.caption = name
    colormap.add_to(m)

def add_categorical_legend(m, title: str, class_to_color: dict, class_labels: dict = None,
                           position: str = "bottomright"):
    pos_css = {
        "topleft":    "top: 10px; left: 10px;",
        "topright":   "top: 10px; right: 10px;",
        "bottomleft": "bottom: 45px; left: 10px;",
        "bottomright":"bottom: 45px; right: 10px;"
    }.get(position, "bottom: 45px; right: 10px;")

    # sort by class code for stable order
    items = []
    for cls in sorted(class_to_color.keys()):
        color = class_to_color[cls]
        label = class_labels.get(cls, str(cls)) if class_labels else str(cls)
        items.append(
            f'<li><span style="display:inline-block;width:12px;height:12px;'
            f'background:{color};margin-right:6px;border:1px solid #555;"></span>'
            f'{label}</li>'
        )
    items_html = "".join(items)

    template = Template(f"""
    {{% macro html(this, kwargs) %}}
    <div style="position: fixed; z-index: 9999; {pos_css} background: white; padding: 8px 10px;
                border: 1px solid #999; border-radius: 4px; box-shadow: 0 1px 4px rgba(0,0,0,.25);">
      <div style="font-weight:600; margin-bottom: 6px;">{title}</div>
      <ul style="list-style:none; padding:0; margin:0; font-size:12px; line-height:1.2;">
        {items_html}
      </ul>
    </div>
    {{% endmacro %}}
    """)
    macro = MacroElement()
    macro._template = template
    m.get_root().add_child(macro)

# ---------- /helpers ----------

In [8]:
df_shape_country = gpd.read_file(r"D:\Work\ADB\adb-school-optimize\data\shapefiles\PNG_country.geojson")
df_shape_district = gpd.read_file(r"D:\Work\ADB\adb-school-optimize\data\shapefiles\PNG_districts.geojson")
df_shape_province = gpd.read_file(r"D:\Work\ADB\adb-school-optimize\data\shapefiles\PNG_provinces.geojson")


In [ ]:


curated = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\png_curated_sec_schools_access_v3_clean.csv")
curated = gpd.GeoDataFrame(curated, geometry=gpd.points_from_xy(curated['Longitude'],  curated['Latitude']))

pop_access_walk = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\pop_access_walk_v2.csv")
pop_no_walk = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\pop_no_walk_v2.csv")
pop_access_drive = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\pop_access_drive_v2.csv")
pop_no_drive = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\pop_no_drive_v2.csv")
pop_access_cycle = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\pop_access_cycle_v2.csv")
pop_no_cycle = pd.read_csv(r"D:\Work\ADB\adb-school-optimize\data\datasets\pop_no_cycle_v2.csv")

roads = gpd.read_file(r"D:\Work\ADB\adb-school-optimize\data\shapefiles\roads_intersect_2026_2.json")

air = gpd.read_file(r"D:\Work\ADB\adb-school-optimize\data\shapefiles\air_quality.geojson")

In [17]:
# district = "National Capital District"
district = 'Talasea'

selected_gadm = df_shape_district[df_shape_district['NAM_2']==district]
clip_geom = selected_gadm[['geometry']]


within_access_walk = pop_access_walk[pop_access_walk['NAM_2'].isin([district ])]
outside_access_walk = pop_no_walk[pop_no_walk['NAM_2'].isin([district])]
within_access_drive = pop_access_drive[pop_access_drive['NAM_2'].isin([district ])]
outside_access_drive = pop_no_drive[pop_no_drive['NAM_2'].isin([district ])]
within_access_cycle = pop_access_cycle[pop_access_cycle['NAM_2'].isin([district ])]
outside_access_cycle = pop_no_cycle[pop_no_cycle['NAM_2'].isin([district])]


district_centroid = clip_geom.geometry.union_all().centroid

road_segments = roads[roads['NAM_2'] == district]

selected_schools = curated[curated['District'] == district]

df_air = air[air['NAM_2'] == district]

In [18]:
from osgeo import gdal

selected_gadm_path = r"D:\Work\ADB\adb-school-optimize\data\shapefiles\selected_gadm.geojson"
land_clipped_tif = r"D:\Work\ADB\adb-school-optimize\data\rasters\Dynamic World LULC_clipped.tif"

selected_gadm.to_file(selected_gadm_path, driver="GeoJSON", index=False)

warp_opts = gdal.WarpOptions(
    format="GTiff",
    cropToCutline=True,
    cutlineDSName=selected_gadm_path,
    creationOptions=[
        "COMPRESS=DEFLATE",
        "PREDICTOR=2",
        "ZLEVEL=6",
        "TILED=YES",
        "BIGTIFF=IF_SAFER"
    ]
)

ds = gdal.Warp(land_clipped_tif, land_tif, options=warp_opts)
ds = None


In [ ]:
if 'opacity' not in within_access_walk.columns or 'opacity' not in outside_access_walk.columns:
    quartile_labels = [0.1, 0.25, 0.5, 1.0]
    both = pd.concat([within_access_walk[['ID','population']], outside_access_walk[['ID','population']]], ignore_index=True)
    both['opacity'] = pd.qcut(both['population'], 4, labels=quartile_labels)
    both_op = both.set_index('ID')['opacity']
    within_access_walk['opacity']  = within_access_walk['ID'].map(both_op)
    outside_access_walk['opacity'] = outside_access_walk['ID'].map(both_op)

if 'opacity' not in within_access_drive.columns or 'opacity' not in outside_access_drive.columns:
    quartile_labels = [0.1, 0.25, 0.5, 1.0]
    both = pd.concat([within_access_drive[['ID','population']], outside_access_drive[['ID','population']]], ignore_index=True)
    both['opacity'] = pd.qcut(both['population'], 4, labels=quartile_labels)
    both_op = both.set_index('ID')['opacity']
    within_access_drive['opacity']  = within_access_drive['ID'].map(both_op)
    outside_access_drive['opacity'] = outside_access_drive['ID'].map(both_op)

if 'opacity' not in within_access_cycle.columns or 'opacity' not in outside_access_cycle.columns:
    quartile_labels = [0.1, 0.25, 0.5, 1.0]
    both = pd.concat([within_access_cycle[['ID','population']], outside_access_cycle[['ID','population']]], ignore_index=True)
    both['opacity'] = pd.qcut(both['population'], 4, labels=quartile_labels)
    both_op = both.set_index('ID')['opacity']
    within_access_cycle['opacity']  = within_access_cycle['ID'].map(both_op)
    outside_access_cycle['opacity'] = outside_access_cycle['ID'].map(both_op)

import time
t0 = time.time()
m = fl.Map(location=[district_centroid.y, district_centroid.x], zoom_start=11)

# Admin boundary (same simplification + no fill)
for _, r in selected_gadm.iterrows():
    # sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.001)
    sim_geo = gpd.GeoSeries(r["geometry"])
    geo_j = fl.GeoJson(data=sim_geo.to_json(), name="Region Boundaries", style_function=lambda x: {"fillColor": None, "color": "orange"})
    fl.Popup(r.get("NAME_2", "Area")).add_to(geo_j)
    geo_j.add_to(m)

# Roads (blue)
fl.GeoJson(
    road_segments[['geometry']].to_json(),
    name='Road segments',
    style_function=lambda f: {'color': 'black', 'weight': 1, 'opacity': 0.5}
).add_to(m)

print(f"Map initialized in {time.time() - t0:.2f} seconds")

# US AQI bands
aqi_bins = [0, 50, 100, 150, 200, 300, 500]
aqi_colors = {
    "Good (0-50)": "#00E400",
    "Moderate (51-100)": "#FFFF00",
    "Unhealthy for Sensitive Groups (101-150)": "#FF7E00",
    "Unhealthy (151-200)": "#FF0000",
    "Very Unhealthy (201-300)": "#8F3F97",
    "Hazardous (301-500)": "#7E0023",
}

def aqi_color(val):
    if pd.isna(val):
        return "#808080"
    if val <= 50:
        return "#00E400"
    elif val <= 100:
        return "#FFFF00"
    elif val <= 150:
        return "#FF7E00"
    elif val <= 200:
        return "#FF0000"
    elif val <= 300:
        return "#8F3F97"
    else:
        return "#7E0023"

mean_aqi_layer = fl.FeatureGroup(name="Average AQI", show=False)

fl.GeoJson(
    df_air.set_geometry("geometry").to_json(),
    name="Air quality",
    style_function=lambda feature: {
        "fillColor": aqi_color(feature["properties"]["aqi_us_mean"]),
        "color": aqi_color(feature["properties"]["aqi_us_mean"]),
        "weight": 1,
        "fillOpacity": 0.6,
        "opacity": 0.8,
    },
    tooltip=fl.GeoJsonTooltip(
        fields=["aqi_us_mean"],
        aliases=["Average AQI"],
        localize=True,
        sticky=True
    )
).add_to(mean_aqi_layer)
mean_aqi_layer.add_to(m)

max_aqi_layer = fl.FeatureGroup(name="Maximum AQI", show=False)
fl.GeoJson(
    df_air.set_geometry("geometry").to_json(),
    name="Air quality",
    style_function=lambda feature: {
        "fillColor": aqi_color(feature["properties"]["aqi_us_max"]),
        "color": aqi_color(feature["properties"]["aqi_us_max"]),
        "weight": 1,
        "fillOpacity": 0.6,
        "opacity": 0.8,
    },
    tooltip=fl.GeoJsonTooltip(
        fields=["aqi_us_max"],
        aliases=["Maximum AQI"],
        localize=True,
        sticky=True
    )
).add_to(max_aqi_layer)
max_aqi_layer.add_to(m)

aqi_legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 240px;
    z-index: 9999;
    background-color: white;
    border: 2px solid grey;
    border-radius: 6px;
    padding: 10px;
    font-size: 14px;
">
<b>US AQI</b><br>
<div><span style="display:inline-block;width:14px;height:14px;background:#00E400;margin-right:8px;"></span>Good (0-50)</div>
<div><span style="display:inline-block;width:14px;height:14px;background:#FFFF00;margin-right:8px;"></span>Moderate (51-100)</div>
<div><span style="display:inline-block;width:14px;height:14px;background:#FF7E00;margin-right:8px;"></span>Unhealthy for SG (101-150)</div>
<div><span style="display:inline-block;width:14px;height:14px;background:#FF0000;margin-right:8px;"></span>Unhealthy (151-200)</div>
<div><span style="display:inline-block;width:14px;height:14px;background:#8F3F97;margin-right:8px;"></span>Very Unhealthy (201-300)</div>
<div><span style="display:inline-block;width:14px;height:14px;background:#7E0023;margin-right:8px;"></span>Hazardous (301-500)</div>
</div>
"""

m.get_root().html.add_child(branca.element.Element(aqi_legend_html))

print(f"AQI layers added in {time.time() - t0:.2f} seconds")


walking_within_layer = fl.FeatureGroup(name="Population Within Access (Walking - 4km)", show=True)
walking_outside_layer = fl.FeatureGroup(name="Population Without Access (Walking - 4km)", show=True)

for _, row in  within_access_walk.iterrows():
    fl.CircleMarker(
        location=[row["ycoord"], row["xcoord"]],
        radius=5,
        fill=True,
        fill_color="green",
        fill_opacity=float(row["opacity"],),
        stroke=False
    ).add_to(walking_within_layer)

for _, row in outside_access_walk.iterrows():
    fl.CircleMarker(
        location=[row["ycoord"], row["xcoord"]],
        radius=5,
        fill=True,
        fill_color="red",
        fill_opacity=float(row["opacity"]),
        stroke=False
    ).add_to(walking_outside_layer)

walking_within_layer.add_to(m)
walking_outside_layer.add_to(m)

print(f"Walking access layers added in {time.time() - t0:.2f} seconds")

cycling_within_layer = fl.FeatureGroup(name="Population Within Access (Cycling - 7 km)", show=False)
cycling_outside_layer = fl.FeatureGroup(name="Population Without Access (Cycling - 7 km)", show=False)
for _, row in  within_access_cycle.iterrows():
    fl.CircleMarker(
        location=[row["ycoord"], row["xcoord"]],
        radius=5,
        fill=True,
        fill_color="green",
        fill_opacity=float(row["opacity"],),
        stroke=False
    ).add_to(cycling_within_layer)

for _, row in outside_access_cycle.iterrows():
    fl.CircleMarker(
        location=[row["ycoord"], row["xcoord"]],
        radius=5,
        fill=True,
        fill_color="red",
        fill_opacity=float(row["opacity"]),
        stroke=False
    ).add_to(cycling_outside_layer)

cycling_within_layer.add_to(m)
cycling_outside_layer.add_to(m)

print(f"Cycling access layers added in {time.time() - t0:.2f} seconds")

driving_within_layer = fl.FeatureGroup(name="Population Within Access (Driving - 10 km)", show=False)
driving_outside_layer = fl.FeatureGroup(name="Population Without Access (Driving - 10 km)", show=False)

for _, row in  within_access_drive.iterrows():
    fl.CircleMarker(
        location=[row["ycoord"], row["xcoord"]],
        radius=5,
        fill=True,
        fill_color="green",
        fill_opacity=float(row["opacity"],),
        stroke=False
    ).add_to(driving_within_layer)

for _, row in outside_access_drive.iterrows():
    fl.CircleMarker(
        location=[row["ycoord"], row["xcoord"]],
        radius=5,
        fill=True,
        fill_color="red",
        fill_opacity=float(row["opacity"]),
        stroke=False
    ).add_to(driving_outside_layer)

driving_within_layer.add_to(m)
driving_outside_layer.add_to(m)

print(f"Driving access layers added in {time.time() - t0:.2f} seconds")

# Elevation (continuous)
# rgba_elev, bounds_elev, elev_vmin, elev_vmax = build_continuous_overlay(
#     elev_tif, geom_clip=clip_geom, cmap_name="terrain", dst_res=None
# )
# fg_elev = fl.FeatureGroup(name="Elevation (m)", show=False)
# fl.raster_layers.ImageOverlay(
#     image=rgba_elev, bounds=bounds_elev, opacity=0.6, mercator_project=False
# ).add_to(fg_elev)
# fg_elev.add_to(m)
# add_colorbar(m, name="Elevation (m)", vmin=elev_vmin, vmax=elev_vmax, cmap_name="terrain")

rgba_lumi, bounds_lumi, lumi_vmin, lumi_vmax = build_continuous_overlay(
    lumi_tif, geom_clip=clip_geom, cmap_name="gray", dst_res=None
)
fg_lumi = fl.FeatureGroup(name="Nighttime luminosity", show=False)
fl.raster_layers.ImageOverlay(
    image=rgba_lumi, bounds=bounds_lumi, opacity=0.6, mercator_project=False
).add_to(fg_lumi)
add_colorbar(m, name="Nighttime luminosity", vmin=lumi_vmin, vmax=lumi_vmax, cmap_name="gray")
fg_lumi.add_to(m)

print(f"Luminosity Raster layers added in {time.time() - t0:.2f} seconds")

# Land cover (categorical)
rgba_lc, bounds_lc = build_categorical_overlay(
    land_clipped_tif,
    class_to_color=landcover_palette,
    geom_clip=None,
    dst_res=0.0005
)
fg_lc = fl.FeatureGroup(name="Land cover", show=False)
fl.raster_layers.ImageOverlay(
    image=rgba_lc, bounds=bounds_lc, opacity=1.0, mercator_project=False
).add_to(fg_lc)
fg_lc.add_to(m)
add_categorical_legend(
    m, title="Land cover",
    class_to_color=landcover_palette,
    class_labels=landcover_labels,
    position="bottomright"
)

print(f"Land cover raster layer added in {time.time() - t0:.2f} seconds")

# Flood depth (continuous)
rgba_flood, bounds_flood, flood_vmin, flood_vmax = build_continuous_overlay(
    flood_tif, geom_clip=clip_geom, cmap_name="Blues", dst_res=None
)
fg_flood = fl.FeatureGroup(name="Flood inundation (m)", show=False)
fl.raster_layers.ImageOverlay(
    image=rgba_flood, bounds=bounds_flood, opacity=0.55, mercator_project=False
).add_to(fg_flood)
fg_flood.add_to(m)
add_colorbar(m, name="Flood inundation (m)", vmin=flood_vmin, vmax=flood_vmax, cmap_name="Blues")

print(f"Flood depth raster layer added in {time.time() - t0:.2f} seconds")

# Schools (blue)
schools_layer = fl.FeatureGroup(name="Schools", show=True)
for i in range(len(selected_schools)):
    row = selected_schools.iloc[i]

    tooltip_html = f"""
    <b>{row['School Name']}</b><br>
    Lat: {round(row['Latitude'], 4)} | Lon: {round(row['Longitude'], 4)}<br>
    Population Served (Walking Access): {row['pop_with_access_walking']}<br>
    Locality: {row['Locality']}<br>
    """

    fl.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=7,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.8,
        tooltip=fl.Tooltip(tooltip_html, sticky=True)
    ).add_to(schools_layer)

schools_layer.add_to(m)

print(f"School layers added in {time.time() - t0:.2f} seconds")

# Layer control (must be added after layers)
fl.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
agg_gb = gpd.read_file(r"D:\Work\ADB\adb-school-optimize\data\datasets\aggregated_district_data.geojson")

In [ ]:
indicator = "Maximum AQI"
indicator = 'Fixed Broadband Download Speed (MB/s)'

labels = {
    "Province": "Province",
    "District": "District",
    indicator: indicator
}

slice_choropleth = agg_gb[['Province', 'District', indicator, 'geometry']]
geojson_data = slice_choropleth.__geo_interface__

is_aqi = "aqi" in indicator.lower()

if is_aqi:
    color_continuous_scale = [
        [0.00, "#00E400"],  # Good
        [0.10, "#00E400"],
        [0.11,  "#FFFF00"],
        [0.20,  "#FFFF00"],
        [0.21, "#FF7E00"],# Unhealthy for Sensitive Groups
        [0.30, "#FF7E00"],
        [0.31, "#FF0000"],  # Unhealthy
        [0.40, "#FF0000"],
        [0.41, "#8F3F97"],  # Very Unhealthy
        [0.50, "#8F3F97"],
        [0.51, "#7E0023"],  # Hazardous
        [0.60, "#7E0023"],
        [0.70, "#7E0023"],
        [0.80, "#7E0023"],
        [0.90, "#7E0023"],
        [1.00, "#7E0023"],
    ]
    range_color = (0, 500)
else:
    color_continuous_scale = "RdYlGn"
    range_color = (slice_choropleth[indicator].min(), slice_choropleth[indicator].max())

fig = px.choropleth_mapbox(
    slice_choropleth,
    geojson=geojson_data,
    locations='District',
    featureidkey='properties.District',
    color=indicator,
    color_continuous_scale=color_continuous_scale,
    range_color=range_color,
    hover_data=['District', 'Province', indicator],
    labels=labels,
    title=f'{indicator} in Papua New Guinea',
    mapbox_style="carto-positron",
    center={"lat": df_shape_country.centroid[0].y, "lon": df_shape_country.centroid[0].x},
    zoom=6,
    opacity=0.7,
    width=1500,
    height=1000
)

fig.add_trace(
    go.Scattermapbox(
        lat=curated["GGMap Latitude"],
        lon=curated["GGMap Longitude"],
        mode="markers",
        name="Schools",
        marker=dict(
            size=6.5,
            color="blue",
            opacity=0.8
        ),
        customdata=curated[[
            "School Name",
            "District Name",
            "Province Name",
            "GGMap Latitude",
            "GGMap Longitude"
        ]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Lat: %{customdata[3]}<br>"
            "Lon: %{customdata[4]}<br>"
            "District: %{customdata[1]}<br>"
            "Province: %{customdata[2]}<extra></extra>"
        )
    )
)


fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    coloraxis_colorbar_title_text=indicator,
    coloraxis_colorbar_title_font=dict(size=15, color="black"),
    title=dict(
        text=f'{indicator} in Papua New Guinea',
        font=dict(size=30, color="black")
    ),
)


In [ ]:
def aqi_color(val):
    if pd.isna(val):
        return "#808080"
    if val <= 50:
        return "#00E400"
    elif val <= 100:
        return "#FFFF00"
    elif val <= 150:
        return "#FF7E00"
    elif val <= 200:
        return "#FF0000"
    elif val <= 300:
        return "#8F3F97"
    else:
        return "#7E0023"

In [ ]:
centroid = df_shape_country.geometry.centroid
center_lat = centroid.iloc[0].y
center_lon = centroid.iloc[0].x

if 'opacity' not in pop_access_walk.columns or 'opacity' not in pop_no_walk.columns:
    quartile_labels = [0.1, 0.25, 0.5, 1.0]
    both = pd.concat([pop_access_walk[['ID','population']], pop_no_walk[['ID','population']]], ignore_index=True)
    both['opacity'] = pd.qcut(both['population'], 4, labels=quartile_labels)
    both_op = both.set_index('ID')['opacity']
    pop_access_walk['opacity']  = pop_access_walk['ID'].map(both_op)
    pop_no_walk['opacity'] = pop_no_walk['ID'].map(both_op)

folium_map = fl.Map(location=[center_lat, center_lon], zoom_start=6, tiles="OpenStreetMap")



for _, r in df_shape_district.iterrows():
    sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.00001)
    geo_j = sim_geo.to_json()
    geo_j = fl.GeoJson(
        data=geo_j,
        style_function=lambda x: {
            "fillColor": "orange",
            "color": "red",
            "weight": 0.4,
            "fillOpacity": 0.0,
        },
    )
    fl.Popup(r["NAM_2"]).add_to(geo_j)
    geo_j.add_to(folium_map)

for i in range(len(curated)):
    row = curated.iloc[i]

    tooltip_html = f"""
    <b>{row['School Name']}</b><br>
    Lat: {row['GGMap Latitude']}<br>
    Lon: {row['GGMap Longitude']}<br>
    District: {row['District Name']}<br>
    Province: {row['Province Name']}
    """

    fl.CircleMarker(
        location=[row['GGMap Latitude'], row['GGMap Longitude']],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.8,
        tooltip=fl.Tooltip(tooltip_html, sticky=True)
    ).add_to(folium_map)


folium_map
